In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text(
    "bronze_table",
    "dbr_dev_ua5816bd.team_tristar_bronze.vehicles_streaming"
)

dbutils.widgets.text(
    "silver_checkpoint",
    "abfss://tristar-team@dlsua5816bd.dfs.core.windows.net/raw/streaming/streaming-metadata/silver-checkpoint"
)

dbutils.widgets.text(
    "silver_table",
    "dbr_dev_ua5816bd.team_tristar_silver.vehicles_streaming"
)

schema_tracking_path = dbutils.widgets.get("silver_checkpoint") + "/schema_tracking"

df = (
    spark.readStream
    .option("schemaTrackingLocation", schema_tracking_path)
    .option("schemaEvolutionMode", "addNewColumns")
    .table(dbutils.widgets.get("bronze_table"))
)

df = df.withColumn(
    'generated',
    F.to_timestamp(F.col('generated'))
)

df = df.withColumn(
    'route_short_name',
    F.when(
        F.trim(F.col('route_short_name')) == '',
        None
    ).otherwise(
        F.trim(F.col('route_short_name'))
    )
)

df = df.withColumn(
    'trip_id',
    F.when(
        F.col('trip_id').try_cast('long') <= 0,
        None
    ).otherwise(
        F.col('trip_id').try_cast('long')
    )
)

df = df.withColumn(
    'route_id',
    F.when(
        F.col('route_id').try_cast('long') <= 0,
        None
    ).otherwise(
        F.col('route_id').try_cast('long')
    )
)

df = df.withColumn(
    'headsign',
    F.when(
        F.trim(F.col('headsign')) == '',
        None
    ).otherwise(
        F.trim(F.col('headsign'))
    )
)

df = df.withColumn(
    'vehicle_code',
    F.when(
        F.trim(F.col('vehicle_code')) == '',
        None
    ).otherwise(
        F.trim(F.col('vehicle_code'))
    )
)

df = df.withColumn(
    'vehicle_service',
    F.when(
        F.trim(F.col('vehicle_service')) == '',
        None
    ).otherwise(
        F.trim(F.col('vehicle_service'))
    )
)

df = df.withColumn(
    'vehicle_id',
    F.when(
        F.col('vehicle_id').try_cast('long') <= 0,
        None
    ).otherwise(
        F.col('vehicle_id').try_cast('long')
    )
)

df = df.withColumn(
    'speed',
    F.when(
        (F.col('speed').try_cast('int') < 0) |
        F.col('speed').try_cast('int').isNull(),
        0
    ).otherwise(
        F.col('speed').try_cast('int')
    )
)

df = df.withColumn(
    'direction',
    F.when(
        (F.col('direction').try_cast('int') < 0) |
        (F.col('direction').try_cast('int') >= 360),
        None
    ).otherwise(
        F.col('direction').try_cast('int')
    )
)

df = df.withColumn(
    'delay',
    F.when(
        F.col('delay').try_cast('int').isNull(),
        0
    ).otherwise(
        F.col('delay').try_cast('int')
    )
)

df = df.withColumn(
    'scheduled_trip_start_time',
    F.when(
        ~F.col('scheduled_trip_start_time').rlike(r'^\d{1,2}:\d{2}:\d{2}$'),
        None
    ).otherwise(
        F.col('scheduled_trip_start_time')
    )
)

df = df.withColumn(
    'latitude',
    F.when(
        (F.col('latitude') < -90) |
        (F.col('latitude') > 90) |
        F.col('latitude').isNull(),
        None
    ).otherwise(
        F.round(F.col('latitude'), 6)
    )
)

df = df.withColumn(
    'longitude',
    F.when(
        (F.col('longitude') < -180) |
        (F.col('longitude') > 180) |
        F.col('longitude').isNull(),
        None
    ).otherwise(
        F.round(F.col('longitude'), 6)
    )
)

df = df.withColumn(
    'gps_quality',
    F.when(
        F.col('gps_quality').try_cast('int') < 0,
        None
    ).otherwise(
        F.col('gps_quality').try_cast('int')
    )
)

query = (
    df.writeStream
        .format("delta")
        .outputMode("append")
        .option("mergeSchema", "true")
        .option("checkpointLocation", dbutils.widgets.get("silver_checkpoint"))
        .trigger(processingTime="20 seconds")
        .toTable(dbutils.widgets.get("silver_table"))
)